# 08 — Ensembling

**Goal**: Combine the out-of-fold (OOF) predictions from our diverse models into a single, stronger meta-model.
We use the predictions from:
- Hand-crafted features (Tuned LightGBM, Tuned XGBoost) [Phase 7]
- K-mer TF-IDF (Logistic Regression, Linear SVM) [Phase 5]

**Verify gates**:
- Load all OOF prediction CSVs
- Plot correlation matrix of predictions (check for diversity)
- Evaluate Simple Averaging vs Stacking (Logistic Regression meta-learner)
- Save the best ensemble weights/model for Phase 9

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from collections import defaultdict

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, roc_curve, auc

plt.style.use('dark_background')
sns.set_palette('viridis')
print('Libraries loaded ✓')

## Step 8.1 — Load OOF Predictions

In [ ]:
train_clusters = pd.read_csv('../data/processed/train_clusters.csv')
y_true = train_clusters['Label'].values
cluster_labels = train_clusters['Cluster'].values

oof_dfs = []
model_aucs = {}

# 1. TF-IDF Models (Phase 5)
if os.path.exists('../data/processed/oof_tfidf.csv'):
    df_tfidf = pd.read_csv('../data/processed/oof_tfidf.csv')
    oof_dfs.append(df_tfidf.drop(columns=['Sequence', 'Label']))
    model_aucs['TF-IDF LogReg'] = roc_auc_score(y_true, df_tfidf['tfidf_lr_pred'])
    model_aucs['TF-IDF SVM']    = roc_auc_score(y_true, df_tfidf['tfidf_svm_pred'])
    print('Loaded TF-IDF OOFs.')

# 2. Tuned Trees (Phase 7)
if os.path.exists('../data/processed/oof_tuned_trees.csv'):
    df_trees = pd.read_csv('../data/processed/oof_tuned_trees.csv')
    oof_dfs.append(df_trees.drop(columns=['Sequence', 'Label']))
    model_aucs['Tuned LGBM']    = roc_auc_score(y_true, df_trees['tuned_lgb_pred'])
    model_aucs['Tuned XGBoost'] = roc_auc_score(y_true, df_trees['tuned_xgb_pred'])
    print('Loaded Tuned Tree OOFs.')

# Combine all
oof_all = pd.concat(oof_dfs, axis=1)
print(f'\nCombined OOF shape: {oof_all.shape}')

print('\nBase Model AUCs:')
for name, score in sorted(model_aucs.items(), key=lambda x: x[1], reverse=True):
    print(f'  {name:<15}: {score:.4f}')

## Step 8.2 — Prediction Correlation Matrix


In [ ]:
corr = oof_all.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='RdYlBu_r', vmin=0.5, vmax=1.0, fmt='.3f', 
            linewidths=1, square=True)
plt.title('Out-Of-Fold Prediction Correlation', fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

## Step 8.3 — Simple Averaging vs Weighted Averaging

In [ ]:
# Simple Mean
mean_preds = oof_all.mean(axis=1)
mean_auc = roc_auc_score(y_true, mean_preds)
print(f'Simple Average AUC: {mean_auc:.4f}')


## Step 8.4 — Stacking (Logistic Regression Meta-Learner)

In [ ]:
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
stack_preds = np.zeros(len(oof_all))
fold_metrics = []

X_meta = oof_all.values

for fold, (train_idx, val_idx) in enumerate(cv.split(X_meta, y_true, groups=cluster_labels)):
    X_tr, y_tr = X_meta[train_idx], y_true[train_idx]
    X_va, y_va = X_meta[val_idx], y_true[val_idx]
    
    meta_model = LogisticRegression(random_state=42, C=0.1) # Strong regularization to prevent overfitting
    meta_model.fit(X_tr, y_tr)
    
    preds = meta_model.predict_proba(X_va)[:, 1]
    stack_preds[val_idx] = preds
    fold_metrics.append(roc_auc_score(y_va, preds))

stack_auc = roc_auc_score(y_true, stack_preds)
print(f'Stacking Meta-Model CV AUC: {stack_auc:.4f} ± {np.std(fold_metrics):.4f}')

# Train final meta-model on all OOF data to inspect weights (importance)
final_meta = LogisticRegression(random_state=42, C=0.1)
final_meta.fit(X_meta, y_true)

plt.figure(figsize=(10, 4))
sns.barplot(x=list(oof_all.columns), y=final_meta.coef_[0], palette='viridis')
plt.title('Meta-Model Weights (Model Importance)', fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

pd.to_pickle(final_meta, '../data/processed/final_meta_model.pkl')
print('Saved 4-feature Meta-Model to data/processed/final_meta_model.pkl')